# Etapa 4. Ajuste das distribuições de probabilidade

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/04_ajuste_fdp.ipynb)

O SPI transforma a chuva acumulada numa probabilidade, e essa probabilidade vem de uma **distribuição** ajustada
aos dados. Se a distribuição for ruim, principalmente nas pontas (as caudas), o SPI erra justamente as secas e os
excessos. Aqui comparamos quatro: **Gama, Gumbel, Log-Normal e Exponencial**.

Para cada **estação**, cada **escala** (3 e 12 meses) e cada **mês do calendário**, o notebook:

1. ajusta as quatro distribuições por máxima verossimilhança;
2. calcula o **AIC** (quanto menor, melhor) e o valor-p do teste de **Kolmogorov-Smirnov (KS)**;
3. nas quatro estações em destaque, refaz os testes KS e **Anderson-Darling (AD)** com a correção por simulação
   (Monte Carlo).

As funções estão em `src/fdp.py`. O capítulo 4 do caderno de aprendizagem mostra um ajuste passo a passo.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# Esta célula:
#   1) define DE ONDE os dados serão lidos (direto do GitHub, no Colab);
#   2) baixa as funções do projeto (pasta src/ do GitHub): fdp.py, spi.py, enos.py, estatistica.py;
#   3) conecta o seu Google Drive e usa a pasta spi_oni_pr/resultados, onde ficam os resultados
#      de uma etapa para a outra.
import sys
import urllib.request
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
REPO = "https://raw.githubusercontent.com/fcoliveira-utfpr/spi_oni_pr/main/"

if Path("../dados").exists():                   # computador do orientador, dentro da pasta do projeto
    DADOS = "../dados/"
    sys.path.append("../src")
else:                                           # Colab
    DADOS = REPO + "dados/"
    Path("src").mkdir(exist_ok=True)
    for arquivo in ["fdp.py", "spi.py", "enos.py", "estatistica.py"]:
        urllib.request.urlretrieve(REPO + "src/" + arquivo, "src/" + arquivo)   # baixa o arquivo
    sys.path.append("src")

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAIDA = Path("/content/drive/MyDrive/spi_oni_pr/resultados")
else:
    SAIDA = Path("../resultados")
SAIDA.mkdir(parents=True, exist_ok=True)
print("Dados lidos de:", DADOS)
print("Resultados gravados em:", SAIDA)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

est = pd.read_csv(DADOS + "estacoes_pr.csv", dtype={"cod": str, "cod_ibge": str})
mensal = pd.read_csv(DADOS + "chuva_mensal_pr.csv", dtype={"cod": str})
meso = gpd.read_file(DADOS + "pr_mesorregioes.geojson")
destaque = est[est["destaque"].notna()]           # as 4 estações em destaque
print(len(est), "estações;", len(mensal), "linhas de chuva mensal")

# Aparência padrão dos gráficos do trabalho (rode uma vez; vale para todos os gráficos do notebook)
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,       # tira a borda de cima
    "axes.spines.right": False,     # tira a borda da direita
    "axes.grid": True,              # grade de fundo...
    "grid.color": "#e6e5e0",        # ...bem clara, para não competir com os dados
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,         # grade atrás das barras e linhas
})

AZUL = "#2a78d6"                    # cor única quando há só uma série
# uma cor fixa para cada estação em destaque (a cor acompanha a estação em todas as figuras)
CORES = {"Santa Helena": "#2a78d6", "Cascavel": "#eb6834", "Foz do Iguaçu": "#1baf7a", "Mercedes": "#eda100"}

from matplotlib.colors import LinearSegmentedColormap

# escala divergente: vermelho (seco / negativo) - cinza (zero) - azul (úmido / positivo)
DIVERGENTE = LinearSegmentedColormap.from_list("div", ["#a3201f", "#e34948", "#f0efec", "#2a78d6", "#184f95"])
# cores das fases do ENOS: El Niño = águas quentes (vermelho), La Niña = águas frias (azul)
CORES_FASE = {"El Niño": "#e34948", "Neutro": "#9a9890", "La Niña": "#2a78d6"}

import json
from scipy import stats
import fdp
import spi

CORES_FDP = {"Gama": "#2a78d6", "Gumbel": "#eb6834", "Log-Normal": "#1baf7a", "Exponencial": "#eda100"}

## 1. Um exemplo: novembro, SPI-3, Santa Helena

A chuva acumulada em 3 meses que termina em novembro é a chuva da **primavera** (setembro + outubro + novembro).
Há um valor por ano: são esses ~39 valores que recebem a distribuição.

In [ ]:
sh = spi.serie_da_estacao(mensal, "2454012")
sh["acum3"] = spi.acumular(sh["chuva_mm"], 3)
primavera = sh.loc[sh["mes"] == 11, "acum3"].dropna()
print(len(primavera), "anos; média =", round(primavera.mean()), "mm")

for nome in fdp.DISTRIBUICOES:
    parametros = fdp.ajustar(primavera, nome)
    print(f"{nome:12s} AIC = {fdp.aic(primavera, nome, parametros):6.1f}   "
          f"p(KS) = {fdp.teste_ks(primavera, nome, parametros):.3f}")

## 2. Todas as estações, escalas e meses

São 370 estações × 2 escalas × 12 meses × 4 distribuições, cerca de 35 mil ajustes. Leva alguns minutos. O resultado
é gravado no Drive (`ajustes_fdp.csv`); se o arquivo já existir, o notebook só o lê, sem refazer as contas.

In [ ]:
ESCALAS = [3, 12]
arquivo = SAIDA / "ajustes_fdp.csv"

if arquivo.exists():
    ajustes = pd.read_csv(arquivo, dtype={"cod": str})
    print("Lido do Drive:", len(ajustes), "linhas")
else:
    linhas = []
    for n, cod in enumerate(est["cod"]):
        serie = spi.serie_da_estacao(mensal, cod)
        for escala in ESCALAS:
            serie["acum"] = spi.acumular(serie["chuva_mm"], escala)
            for m in range(1, 13):
                valores = serie.loc[serie["mes"] == m, "acum"].dropna()
                for nome in fdp.DISTRIBUICOES:
                    parametros = fdp.ajustar(valores, nome)
                    linhas.append({"cod": cod, "escala": escala, "mes": m, "distribuicao": nome,
                                   "aic": fdp.aic(valores, nome, parametros),
                                   "p_ks": fdp.teste_ks(valores, nome, parametros)})
        if n % 50 == 0:
            print("estação", n, "de", len(est))
    ajustes = pd.DataFrame(linhas)
    ajustes.to_csv(arquivo, index=False)
    print("Pronto:", len(ajustes), "ajustes")

## 3. Qual distribuição ajusta melhor? (Tabela 4 do TCC)

Para cada caso (estação × escala × mês), a **melhor** é a de menor AIC. Contamos em quantos % dos casos cada
distribuição foi a melhor e em quantos % foi **aprovada** pelo KS (p ≥ 0,05).

In [ ]:
# para cada caso, a linha de menor AIC
melhor = ajustes.loc[ajustes.groupby(["cod", "escala", "mes"])["aic"].idxmin()]
pct_melhor = melhor.groupby("escala")["distribuicao"].value_counts(normalize=True).unstack() * 100

ajustes["aprovada_ks"] = ajustes["p_ks"] >= 0.05
pct_ks = ajustes.groupby(["escala", "distribuicao"])["aprovada_ks"].mean().unstack() * 100

tabela4 = pd.concat({"melhor pelo AIC (%)": pct_melhor, "aprovada no KS (%)": pct_ks}, axis=1).round(1)
tabela4.to_csv(SAIDA / "tabela04_ajuste_fdp.csv")
tabela4

O KS clássico "aprova" com facilidade porque os parâmetros foram estimados dos próprios dados. Nas quatro estações
em destaque, refazemos KS e AD com a **correção por simulação** (Monte Carlo, 99 simulações). O AD dá mais peso às
caudas, onde estão as secas e os excessos. Leva uns minutos.

In [ ]:
linhas = []
for i, linha in destaque.iterrows():
    serie = spi.serie_da_estacao(mensal, linha["cod"])
    for escala in ESCALAS:
        serie["acum"] = spi.acumular(serie["chuva_mm"], escala)
        for m in range(1, 13):
            valores = serie.loc[serie["mes"] == m, "acum"].dropna()
            for nome in fdp.DISTRIBUICOES:
                p = fdp.testes_monte_carlo(valores, nome, n_simulacoes=99)
                linhas.append({"estacao": linha["municipio"], "escala": escala, "mes": m, "distribuicao": nome,
                               "p_ks_mc": p["ks"], "p_ad_mc": p["ad"]})
mc = pd.DataFrame(linhas)
mc.to_csv(SAIDA / "testes_monte_carlo_destaque.csv", index=False)

mc["aprovada_ks"] = mc["p_ks_mc"] >= 0.05
mc["aprovada_ad"] = mc["p_ad_mc"] >= 0.05
(mc.groupby(["escala", "distribuicao"])[["aprovada_ks", "aprovada_ad"]].mean() * 100).round(0)

## 4. Figura 4 do TCC: histograma e curvas

Santa Helena, chuva da primavera (acumulada em 3 meses até novembro) e chuva do ano (acumulada em 12 meses até
dezembro), com as quatro curvas ajustadas por cima. O histograma está em **densidade** (a área total das barras
vale 1), para poder comparar com as curvas.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
for ax, (escala, mes, titulo) in zip(eixos, [(3, 11, "SPI-3: setembro a novembro"), (12, 12, "SPI-12: janeiro a dezembro")]):
    sh["acum"] = spi.acumular(sh["chuva_mm"], escala)
    valores = sh.loc[sh["mes"] == mes, "acum"].dropna()
    ax.hist(valores, bins=10, density=True, color="#e6e5e0", edgecolor="white")
    x = np.linspace(1, valores.max() * 1.15, 300)              # 300 pontos para desenhar as curvas
    for nome in fdp.DISTRIBUICOES:
        parametros = fdp.ajustar(valores, nome)
        ax.plot(x, fdp.DISTRIBUICOES[nome].pdf(x, *parametros), color=CORES_FDP[nome], linewidth=2, label=nome)
    ax.set_title(f"Santa Helena, {titulo}", fontsize=10)
    ax.set_xlabel("Chuva acumulada (mm)")
    ax.set_ylabel("Densidade")
eixos[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(SAIDA / "fig04_ajuste_fdp_santa_helena.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. A distribuição adotada no SPI (ponto de decisão)

A proposta padrão: para cada escala, a distribuição que foi a **melhor pelo AIC no maior número de casos**, a mesma
para todas as estações (assim os SPIs são comparáveis entre si). A escolha é gravada em `fdp_escolhida.json`, que o
notebook 05 lê. **Discuta com o orientador antes de seguir**; se a decisão for outra, mude o dicionário na célula
abaixo e rode de novo.

In [ ]:
escolhida = {}
for escala in ESCALAS:
    escolhida[str(escala)] = pct_melhor.loc[escala].idxmax()     # a de maior % de "melhor"

# para mudar a escolha, descomente e edite a linha abaixo:
# escolhida = {"3": "Gama", "12": "Gama"}

with open(SAIDA / "fdp_escolhida.json", "w") as f:
    json.dump(escolhida, f)
print("Distribuição adotada no SPI:", escolhida)

---

## O que observar

- A **Exponencial**, com um parâmetro só, deve ser a pior: ela não tem como acompanhar a forma da chuva acumulada.
- Gama e Log-Normal costumam disputar o primeiro lugar; a Gumbel pode ir bem nos meses chuvosos. Compare com
  Blain e Meschiatti, Silva, Í. N. et al. (2013) e Oliveira et al. (2025).
- Repare como o AD corrigido reprova mais que o KS clássico: é o efeito das caudas.

## Exercícios

**Exercício 1.** Em quais meses a distribuição escolhida foi menos vezes a melhor? Use a tabela `melhor` e
`groupby(["escala", "mes"])`. Por que esses meses seriam mais difíceis?

In [ ]:
# escreva sua resposta aqui

**Exercício 2.** Repita a figura 4 para Cascavel (`"2453056"`). As curvas ficam parecidas com as de Santa Helena?

In [ ]:
# escreva sua resposta aqui

---

**Entregar.** `fig04`, `tabela04`, `fdp_escolhida.json` (decidida com o orientador) e os exercícios.
**O que escrever:** item 5.3 do TCC2 e completar a orientação em azul do item 4.3.